# Notebook 24 — Entrenamiento Modelo C (YOLOv8m-seg)

**TFM — Sistema de Detección de Amenazas Armadas en Vídeo**  
Oliver Legarreta García · Universitat Oberta de Catalunya

---

## Objetivo

Entrenar **Modelo C** — un detector de armas basado en **segmentación de instancias** en lugar de detección por bounding box — y compararlo con Modelo B en el dataset GAR.

## Comparativa de modelos

| Modelo | Arquitectura | Task | Dataset entrenamiento | Negativos difíciles |
|--------|-------------|------|-----------------------|--------------------|
| **A** | yolov8m | Detección | Dataset armas original | Sin |
| **B** | yolov8m | Detección | Dataset armas + COCO | ~3.000 COCO |
| **C** | yolov8m-seg | **Segmentación** | Roboflow gun-project + LVIS | 108 LVIS |

## Dataset Modelo C

- **Positivos:** 8.246 imágenes con máscaras de armas (Roboflow `gun-project-nzlce`)
- **Negativos difíciles:** 108 imágenes LVIS con máscaras de teléfonos, botellas y mandos
- **Clase única:** `gun` (clase 0)
- **Formato:** YOLOv8 segmentación (polígonos normalizados)

---
## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install ultralytics
import ultralytics
ultralytics.checks()
print('✅ Ultralytics instalado')

In [ ]:
import os
import shutil
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# ── CONFIG ────────────────────────────────────────────────────────────────────
GUN_DS     = '/content/drive/MyDrive/TFM/datasets/gun_segmentation_roboflow'
LVIS_DS    = '/content/drive/MyDrive/TFM/datasets/lvis_negatives'
COMBINED   = '/content/dataset_modelo_c'
OUT_DIR    = '/content/drive/MyDrive/TFM/experiments/weapon_seg/yolov8m_seg_C'

EPOCHS     = 50
IMG_SIZE   = 640
BATCH      = 16
PATIENCE   = 15   # early stopping

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('✅ Config cargada')
print(f'   Epochs:   {EPOCHS}')
print(f'   ImgSize:  {IMG_SIZE}')
print(f'   Batch:    {BATCH}')

---
## 1. Verificar datasets

In [ ]:
# Verificar dataset Roboflow
print('=== Dataset Roboflow (gun_segmentation_roboflow) ===')
for split in ['train', 'valid', 'test']:
    img_dir = Path(GUN_DS) / split / 'images'
    lbl_dir = Path(GUN_DS) / split / 'labels'
    imgs = list(img_dir.glob('*'))
    lbls = list(lbl_dir.glob('*.txt'))
    print(f'  {split:<6}: {len(imgs):>5} imágenes | {len(lbls):>5} labels')

print()
print('=== Dataset LVIS negativos ===')
lvis_imgs = list((Path(LVIS_DS) / 'images' / 'train').glob('*.jpg'))
lvis_lbls = list((Path(LVIS_DS) / 'labels' / 'train').glob('*.txt'))
print(f'  train: {len(lvis_imgs):>5} imágenes | {len(lvis_lbls):>5} labels')

# Verificar que los labels LVIS son polígonos (más de 5 valores por línea)
sample_lbl = next(iter(lvis_lbls))
first_line = sample_lbl.read_text().strip().split('\n')[0]
n_coords = len(first_line.split()) - 1  # -1 por la clase
print(f'  Formato label LVIS: {n_coords} coordenadas por polígono ({'✅ segmentación' if n_coords > 8 else '❌ bbox'})')

---
## 2. Construir dataset combinado

In [ ]:
# Estructura del dataset combinado:
# /content/dataset_modelo_c/
#   train/images/  ← gun train + LVIS negativos
#   train/labels/
#   valid/images/  ← gun valid (sin LVIS)
#   valid/labels/
#   test/images/   ← gun test
#   test/labels/
#   data.yaml

if Path(COMBINED).exists():
    shutil.rmtree(COMBINED)
    print('  Carpeta combinada anterior eliminada')

for split in ['train', 'valid', 'test']:
    (Path(COMBINED) / split / 'images').mkdir(parents=True)
    (Path(COMBINED) / split / 'labels').mkdir(parents=True)

# Copiar gun dataset
print('Copiando dataset Roboflow...')
for split, src_split in [('train','train'), ('valid','valid'), ('test','test')]:
    src_img = Path(GUN_DS) / src_split / 'images'
    src_lbl = Path(GUN_DS) / src_split / 'labels'
    dst_img = Path(COMBINED) / split / 'images'
    dst_lbl = Path(COMBINED) / split / 'labels'
    n = 0
    for img in src_img.glob('*'):
        shutil.copy2(img, dst_img / img.name)
        lbl = src_lbl / (img.stem + '.txt')
        if lbl.exists():
            shutil.copy2(lbl, dst_lbl / lbl.name)
        n += 1
    print(f'  {split}: {n} imágenes copiadas')

# Añadir LVIS negativos al train
# Los negativos LVIS tienen clase 0 (no_weapon)
# Necesitamos asignarles clase 1 para que el modelo aprenda a NO detectarlos como arma
# En YOLOv8, imágenes con label vacío = negativo puro
# Usamos archivos de label vacíos para indicar que no hay armas
print('\nAñadiendo negativos LVIS al train...')
lvis_img_dir = Path(LVIS_DS) / 'images' / 'train'
dst_img = Path(COMBINED) / 'train' / 'images'
dst_lbl = Path(COMBINED) / 'train' / 'labels'
n_lvis = 0
for img in lvis_img_dir.glob('*.jpg'):
    shutil.copy2(img, dst_img / img.name)
    # Label vacío = imagen sin armas (negativo puro)
    (dst_lbl / (img.stem + '.txt')).write_text('')
    n_lvis += 1
print(f'  train: {n_lvis} negativos LVIS añadidos (labels vacíos)')

# Verificar totales
print()
print('=== Dataset combinado final ===')
for split in ['train', 'valid', 'test']:
    imgs = list((Path(COMBINED) / split / 'images').glob('*'))
    lbls = list((Path(COMBINED) / split / 'labels').glob('*.txt'))
    print(f'  {split:<6}: {len(imgs):>5} imágenes | {len(lbls):>5} labels')

In [ ]:
# Generar data.yaml
data_yaml = {
    'path': COMBINED,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc': 1,
    'names': {0: 'gun'},
}

yaml_path = Path(COMBINED) / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print('✅ data.yaml generado:')
print(open(yaml_path).read())

---
## 3. Verificación visual del dataset

In [ ]:
import cv2
import numpy as np

def visualize_seg_sample(img_path, lbl_path, ax, title=''):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    overlay = img.copy()

    if lbl_path.exists() and lbl_path.stat().st_size > 0:
        for line in lbl_path.read_text().strip().split('\n'):
            parts = line.strip().split()
            if len(parts) < 7: continue
            coords = list(map(float, parts[1:]))
            pts = np.array([[coords[i]*w, coords[i+1]*h]
                            for i in range(0, len(coords), 2)], dtype=np.int32)
            cv2.fillPoly(overlay, [pts], (255, 100, 0))
            cv2.polylines(img, [pts], True, (255, 200, 0), 2)
        img = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)

    ax.imshow(img)
    ax.set_title(title, fontsize=8)
    ax.axis('off')

# Mostrar 6 positivos y 3 negativos LVIS
train_img_dir = Path(COMBINED) / 'train' / 'images'
train_lbl_dir = Path(COMBINED) / 'train' / 'labels'

gun_imgs  = [f for f in sorted(train_img_dir.glob('*'))
             if not f.name.startswith('lvis_')][:6]
lvis_imgs = [f for f in sorted(train_img_dir.glob('lvis_*.jpg'))][:3]

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for i, img_path in enumerate(gun_imgs[:5]):
    lbl = train_lbl_dir / (img_path.stem + '.txt')
    visualize_seg_sample(img_path, lbl, axes[0, i], f'GUN — {img_path.name[:12]}')
for i, img_path in enumerate(lvis_imgs[:3]):
    lbl = train_lbl_dir / (img_path.stem + '.txt')
    visualize_seg_sample(img_path, lbl, axes[1, i], f'NEG — {img_path.name[:12]}')
axes[1, 3].axis('off')
axes[1, 4].axis('off')

plt.suptitle('Muestra del dataset Modelo C — positivos (arriba) y negativos LVIS (abajo)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/muestra_dataset_C.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Muestra guardada')

---
## 4. Entrenamiento Modelo C

In [ ]:
from ultralytics import YOLO

# yolov8m-seg: mismo tamaño que Modelo B (yolov8m) pero variante segmentación
model = YOLO('yolov8m-seg.pt')

print('Iniciando entrenamiento Modelo C...')
print(f'  Arquitectura: yolov8m-seg')
print(f'  Epochs:       {EPOCHS}')
print(f'  Batch:        {BATCH}')
print(f'  ImgSize:      {IMG_SIZE}')
print(f'  Dataset:      {COMBINED}')
print()

results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=PATIENCE,
    device='cuda',
    project=OUT_DIR,
    name='train',
    exist_ok=True,
    val=True,
    save=True,
    plots=True,
    # Augmentaciones — mismas que Modelo B para comparación justa
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=0.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
)

print('\n✅ Entrenamiento completado')

---
## 5. Resultados del entrenamiento

In [ ]:
# Curvas de entrenamiento
results_dir = Path(OUT_DIR) / 'train'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
results_csv = results_dir / 'results.csv'
if results_csv.exists():
    import pandas as pd
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    ax = axes[0]
    for col in [c for c in df.columns if 'loss' in c.lower() and 'train' in c.lower()]:
        ax.plot(df['epoch'], df[col], label=col.strip())
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Train Loss')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

    ax = axes[1]
    for col in [c for c in df.columns if 'map' in c.lower()]:
        ax.plot(df['epoch'], df[col], label=col.strip())
    ax.set_xlabel('Epoch')
    ax.set_ylabel('mAP')
    ax.set_title('Validation mAP')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Curvas de entrenamiento — Modelo C (yolov8m-seg)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/curvas_entrenamiento_C.png', dpi=120, bbox_inches='tight')
plt.show()

# Métricas finales
print('\n=== MÉTRICAS FINALES MODELO C ===')
last = df.iloc[-1]
for col in df.columns:
    if any(k in col.lower() for k in ['map', 'precision', 'recall', 'loss']):
        print(f'  {col.strip():<40}: {last[col]:.4f}')

---
## 6. Guardar pesos en Drive

In [ ]:
best_weights = results_dir / 'weights' / 'best.pt'
dst_weights  = Path(OUT_DIR) / 'weights' / 'best.pt'
dst_weights.parent.mkdir(parents=True, exist_ok=True)

if best_weights.exists():
    shutil.copy2(best_weights, dst_weights)
    size_mb = dst_weights.stat().st_size / 1024 / 1024
    print(f'✅ Pesos guardados: {dst_weights}')
    print(f'   Tamaño: {size_mb:.1f} MB')
else:
    print('❌ No se encontraron pesos — revisar entrenamiento')

print()
print('Próximo paso: Notebook 25 — Evaluación Modelo C sobre dataset GAR')
print('Comparativa final: Modelo B (detección) vs Modelo C (segmentación)')

---
## Notas para el Notebook 25 (evaluación)

**Modelo C usa `yolov8m-seg`** — la inferencia devuelve tanto bounding boxes como máscaras de segmentación. Para la evaluación clip-level del GAR se usará el mismo criterio que Modelo B: si hay al menos una detección con conf ≥ 0.25 en el frame, el frame se cuenta como positivo.

**Comparativa final esperada:**

| Modelo | Arquitectura | F1 GAR | FP | FN |
|--------|-------------|--------|-----|-----|
| B | yolov8m (detección) | 0.7949 | 48 | 16 |
| C | yolov8m-seg (segmentación) | ? | ? | ? |

**Hipótesis:** Modelo C debería reducir FP en N6–N9 (teléfono) y N10–N11 (botella) gracias a que aprendió la forma exacta del arma mediante segmentación, siendo más preciso en la discriminación morfológica.